In [30]:
import json
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from collections import defaultdict
import json

In [31]:
# Define sets
A = ["Measles", "Mumps", "Rubella", "Diphtheria", "Tetanus", "Pertussis", "Hepatitis_B", "Hib", "Polio", "HPV", "Rotavirus", "PCV"]

V = ["M", "MR", "MMR", "TT", "HepB", "Hib", "IPV", "OPV", "DT", "Td", "DTwP", "DTwP-Hib", "Penta", "Hexa", "HPV", "Rotavirus", "PCV"]

A_v = {
    "M": ["Measles"],
    "MR": ["Measles", "Rubella"],
    "MMR": ["Measles", "Mumps", "Rubella"],
    "TT": ["Tetanus"],
    "HepB": ["Hepatitis_B"],
    "Hib": ["Hib"],
    "IPV": ["Polio"],
    "OPV": ["Polio"],
    "DT": ["Diphtheria", "Tetanus"],
    "Td": ["Diphtheria", "Tetanus"],
    "DTwP": ["Diphtheria", "Tetanus", "Pertussis"],
    "DTwP-Hib": ["Diphtheria", "Tetanus", "Pertussis", "Hib"],
    "Penta": ["Diphtheria", "Tetanus", "Pertussis", "Hepatitis_B", "Hib"],
    "Hexa": ["Diphtheria", "Tetanus", "Pertussis", "Hepatitis_B", "Hib", "Polio"],
    "HPV": ["HPV"],
    "Rotavirus": ["Rotavirus"],
    "PCV": ["PCV"]
}

P = [
    "AJ_Vaccines",
    "BB_NCIPD",
    "China_National",
    "Bharat_Biotech",
    "Bilthoven",
    "Biological_E",
    "GSK",
    "Haffkine_Bio",
    "LG_Chem",
    "Merck_Sharp",
    "Panacea_Biotec",
    "PT_Bio",
    "Sanofi",
    "Serum_Institute",
    "Pfizer"
]

P_v = {
    "M": ["Serum_Institute", "PT_Bio"],
    "MR": ["Serum_Institute", "Biological_E"],
    "MMR": ["Serum_Institute", "GSK"],
    "TT": ["Serum_Institute", "PT_Bio", "BB_NCIPD", "Biological_E"],
    "HepB": ["Serum_Institute", "LG_Chem"],
    "Hib": ["Serum_Institute"],
    "IPV": ["LG_Chem", "AJ_Vaccines", "Bilthoven", "Sanofi"],
    "OPV": ["Serum_Institute", "PT_Bio", "GSK", "Sanofi", "Panacea_Biotec", "China_National", "Bharat_Biotech", "Haffkine_Bio"],
    "DT": ["PT_Bio", "BB_NCIPD"],
    "Td": ["Serum_Institute", "PT_Bio", "BB_NCIPD", "Biological_E"],
    "DTwP": ["Serum_Institute", "Biological_E"],
    "DTwP-Hib": ["Serum_Institute"],
    "Penta": ["Serum_Institute", "PT_Bio", "Biological_E", "LG_Chem", "Panacea_Biotec"],
    "Hexa": ["Sanofi"],
    "HPV": ["GSK", "Merck_Sharp", "China_National"],
    "Rotavirus": ["Serum_Institute", "GSK", "Bharat_Biotech"],
    "PCV": ["Serum_Institute", "GSK", "Pfizer"]
}

In [32]:
# Define constants
beta = 10.0  

tmin = 1
tmax = 10

max_tender_length = 5

unit = 1000

Δ = [i for i in range(1, max_tender_length + 1)]

# Generate time periods
T = [*range(tmin, tmax + 1)]

# Calculate delta values
delta = {t: (1 + 0.03) ** t for t in T}

#  Tender cost
g = {t: 1e8/unit for t in T}

# Cost of expanding capacity for each producer
gamma = {p: 1e8/unit for p in P}  

# Inventory holding cost
h = {v: 0.01 for v in V}  

F_time_set = []
for t in T:
    for tau in T:
        if tau >= t:
            if (tau - t + 1) in Δ:
                F_time_set.append((t, tau))

In [33]:
# Import data
filename = r"F:\Git\Vaccine_Tender\DATA\Starting_point.xlsx"
starting_points_file_F = pd.read_excel(filename, sheet_name="F_start")

starting_points_vect_F = [
    (row['Antigen'], (row['Starting'], row['Ending']))
    for _, row in starting_points_file_F.iloc[0:].iterrows()
]

# Scenario probabilities
with open(r"F:\Git\Vaccine_Tender\DATA\ORIGINAL_PAIRS\data\scenario_pair_probabilities_new.json", 'r') as f:
    probabilities = json.load(f)

# Import model results
file_path = r"F:\Git\Vaccine_Tender\L-shaped\Final_codes\results\unicef-gavi 5 x7 scenarios\Phase2_L_results_T_10_delta_5_scen_35_trial_1_inv_1_cap._1_cap.inc._1.json"
with open(file_path, 'r') as file:
    data = json.load(file)

In [34]:
# Read in and transform price data to dict
file_path = r"F:\Git\Vaccine_Tender\L-shaped\Final_codes\data\Vaccine_price_data.xlsx"
xlsx = pd.ExcelFile(file_path)
sheet_names = xlsx.sheet_names[2:]
vaccine_dict = {}

for sheet in sheet_names:
    df = pd.read_excel(file_path, sheet_name=sheet)
    sheet_dict = {}
    for _, row in df.iterrows():
        producer = row['Unnamed: 0'] if 'Unnamed: 0' in row else None
        if producer:
            if producer not in sheet_dict:
                sheet_dict[producer] = {}
            for col in df.columns:
                if isinstance(col, int):
                    sheet_dict[producer][col] = row[col]
    vaccine_dict[sheet] = sheet_dict

vaccine_dict_sample = {sheet: list(vaccine_dict[sheet].items()) for sheet in vaccine_dict}
modified_dict = {}

for key, value in vaccine_dict_sample.items():
    new_key = key.split()[0]
    modified_dict[new_key] = value

vaccine_price_dict = modified_dict

for vaccine, producers_list in vaccine_price_dict.items():
    producers_dict = dict(producers_list)
    vaccine_price_dict[vaccine] = producers_dict

In [35]:
# Calculate average price per year per vaccine
avg_prices_per_period = {}

for vaccine, producers in vaccine_price_dict.items():
    avg_prices_per_period[vaccine] = {}
    
    prices_by_time = {}
    for producer, values in producers.items():
        for time, price in values.items():
            if isinstance(price, (int, float)):
                if time not in prices_by_time:
                    prices_by_time[time] = []
                prices_by_time[time].append(price)

    for time, prices in prices_by_time.items():
        avg_prices_per_period[vaccine][time] = sum(prices) / len(prices) if prices else 0


In [36]:
# OBJECTIVE FUNCTION COMPONENTS
##########################

# 1. Calculate Tender costs - g[t] * F[a,t,tau] / delta[t]
result = {}
for antigen, data_t in data.get("F", {}).items():
    result[antigen] = {}
    for t, data_tau in data_t.items():
        t = int(t)
        if t in T:
            result[antigen][t] = {}
            for tau, value in data_tau.items():
                result[antigen][t][tau] = g[t] * value / delta[t]

F_OBJ_Value = 0
for antigen, data_t in result.items():
    for t, data_tau in data_t.items():
        F_OBJ_Value += sum(data_tau.values())

print(f"F Objective cost: {F_OBJ_Value}")


F Objective cost: 3866759.9511937974


In [37]:
# 2. Calculate Capacity Extension Costs - gamma[p] * L[p,t] / delta[t]
L_data = data.get("L", {})
transformed_L_data = {}

for producer, years in L_data.items():
    for year, value in years.items():
        if year not in transformed_L_data:
            transformed_L_data[year] = {}
        transformed_L_data[year][producer] = value

result_after_gamma = {}
for year, producers in transformed_L_data.items():
    result_after_gamma[year] = {}
    for producer, value in producers.items():
        result_after_gamma[year][producer] = value * gamma[producer] / delta[int(year)]

L_OBJ_Value = 0
for year, producers in result_after_gamma.items():
    for producer, value in producers.items():
        L_OBJ_Value += value

print(f"L Objective Value: {L_OBJ_Value}")

L Objective Value: 10889020.720960306


In [38]:
# 3. Calculate missed doses by scenario - Beta * S[a,t,omega] / delta[t]
def process_scenarios(S_data, beta, delta):
    S_data_by_scenario = defaultdict(lambda: defaultdict(lambda: defaultdict(dict)))
    for antigen, year_data in S_data.items():
        for year, scenario_data in {y: d for y, d in year_data.items() if y != '0'}.items():
            for scenario, value in scenario_data.items():
                S_data_by_scenario[scenario][year][antigen] = value

    S_data_by_scenario = dict(S_data_by_scenario)

    S_data_by_scenario_scaled = defaultdict(lambda: defaultdict(lambda: defaultdict(dict)))
    for scenario, years in S_data_by_scenario.items():
        for year, antigens in years.items():
            for antigen in antigens.keys():
                S_data_by_scenario_scaled[scenario][year][antigen] = (
                    S_data_by_scenario[scenario][year][antigen] * beta / delta[int(year)]
                )

    scenario_sums = {}
    for scenario, years in S_data_by_scenario_scaled.items():
        scenario_sum = sum(
            value
            for year in years.values()
            for value in year.values()
            if isinstance(value, (int, float))
        )
        scenario_sums[scenario] = scenario_sum

    return scenario_sums

S_data = data['S']
scenario_sums_S = process_scenarios(S_data, beta, delta)

S_OBJ_Values = {k: probabilities[k] * scenario_sums_S[k] for k in probabilities}
S_OBJ_Value = sum(S_OBJ_Values.values())
print(f"Missed Dose OBJ Value: {(S_OBJ_Value)}")


Missed Dose OBJ Value: 120901490.49312574


In [39]:
# 4. Calculate doses purchased - r[v,p,t] * X[v,p,t,omega] / delta[t]
X_data = data['X']

def reorganize_x_data(x_data):
    reorganized_x_data = {}
    for v, producers in x_data.items():
        for p, time_periods in producers.items():
            for t, scenarios in time_periods.items():
                for omega, value in scenarios.items():
                    if omega not in reorganized_x_data:
                        reorganized_x_data[omega] = {}
                    if v not in reorganized_x_data[omega]:
                        reorganized_x_data[omega][v] = {}
                    if p not in reorganized_x_data[omega][v]:
                        reorganized_x_data[omega][v][p] = {}
                    reorganized_x_data[omega][v][p][t] = value
    return reorganized_x_data

reorganized_x_data = reorganize_x_data(X_data)

def calculate_scenario_results(reorganized_x_data, vaccine_price_dict, delta):
    scenario_results = {}
    for omega, vaccines in reorganized_x_data.items():
        scenario_results[omega] = {}
        for v, producers in vaccines.items():
            if v in vaccine_price_dict:
                for p, time_periods in producers.items():
                    if p in vaccine_price_dict[v]:
                        for t, value in time_periods.items():
                            t_int = int(t)
                            if t_int in vaccine_price_dict[v][p]:
                                scenario_results[omega][(v, p, t)] = (
                                    vaccine_price_dict[v][p][t_int] * value
                                ) / delta[t_int]
    return scenario_results

scenario_results = calculate_scenario_results(reorganized_x_data, vaccine_price_dict, delta)

# OLD CODE - This caused TypeError
# X_OBJ_Values = {k: probabilities[k] * scenario_results[k] for k in probabilities}
# X_OBJ_Value = sum(X_OBJ_Values.values())

# NEW CODE - First sum values for each scenario, then multiply by probability
X_scenario_sums = {}
for scenario, entries in scenario_results.items():
    scenario_sum = sum(entries.values())
    X_scenario_sums[scenario] = scenario_sum

# Then multiply by probabilities
X_OBJ_Values = {k: probabilities[k] * X_scenario_sums.get(k, 0) for k in probabilities}
X_OBJ_Value = sum(X_OBJ_Values.values())
print(f"Vaccines Purchased OBJ Value: {X_OBJ_Value}")

Vaccines Purchased OBJ Value: 16759335.745243901


In [49]:
# 5. Calculate inventory holding costs - h[v] * r_bar[v,t] * I[v,t,omega] / delta[t]
import math

# Get inventory data and transform it
I_data = data.get("I", {})
reversed_data = defaultdict(lambda: defaultdict(dict))
for vaccine, years in I_data.items():
    for year, scenarios in years.items():
        for scenario, value in scenarios.items():
            reversed_data[scenario][year][vaccine] = value

# Remove year zero
def remove_year_zero(data):
    for scenario, years in list(data.items()):
        if isinstance(years, defaultdict) or isinstance(years, dict):
            for year in list(years.keys()):
                if year == '0':
                    del years[year]
    return data

reversed_remove_start_I = remove_year_zero(reversed_data)

# Calculate inventory costs with robust error handling
result = defaultdict(lambda: defaultdict(lambda: defaultdict(float)))
for scenario, years in reversed_remove_start_I.items():
    for year, vaccines in years.items():
        try:
            year_int = int(year)
            delta_value = delta.get(year_int)
            
            if delta_value is None or delta_value == 0:
                continue
                
            for vaccine, value in vaccines.items():
                # Skip NaN values
                if math.isnan(value):
                    continue
                
                # Calculate only if we have price data
                if vaccine in avg_prices_per_period and year_int in avg_prices_per_period[vaccine]:
                    avg_price = avg_prices_per_period[vaccine][year_int]
                    h_value = h.get(vaccine, 0.01)
                    
                    try:
                        calculated = (value * avg_price * h_value) / delta_value
                        
                        # Only keep valid results
                        if not math.isnan(calculated) and not math.isinf(calculated):
                            result[scenario][year][vaccine] = calculated
                    except Exception:
                        pass
        except Exception:
            continue

# Sum by scenario
scenario_sums_H = {}
for scenario, years in result.items():
    scenario_sum = 0
    for year, vaccines in years.items():
        for vaccine, value in vaccines.items():
            scenario_sum += value
    scenario_sums_H[scenario] = scenario_sum

# Calculate weighted inventory cost
I_OBJ_Values = {}
for k in probabilities:
    if k in scenario_sums_H:
        I_OBJ_Values[k] = probabilities[k] * scenario_sums_H[k]
    else:
        I_OBJ_Values[k] = 0

I_OBJ_Value = sum(I_OBJ_Values.values())
print(f"Inventory Holding OBJ Value: {I_OBJ_Value}")

Inventory Holding OBJ Value: 123271.38689212693


In [50]:
# TOTAL OBJECTIVE FUNCTION VALUE
# OLD CODE
# print(f"OBJ Value: {S_OBJ_Value}") 

# NEW CODE - Complete objective function
Total_OBJ_Value = F_OBJ_Value + L_OBJ_Value + S_OBJ_Value + X_OBJ_Value + I_OBJ_Value
print(f"Total OBJ Value: {Total_OBJ_Value}")

Total OBJ Value: 152539878.29741588


In [48]:
# Robust implementation of inventory cost calculation
import math

result = defaultdict(lambda: defaultdict(lambda: defaultdict(float)))
total_entries = 0
valid_entries = 0
zero_entries = 0
non_zero_entries = 0

for scenario, years in reversed_remove_start_I.items():
    for year, vaccines in years.items():
        try:
            year_int = int(year)
            delta_value = delta.get(year_int)
            
            if delta_value is None or delta_value == 0:
                print(f"Warning: Invalid delta for year {year}")
                continue
                
            for vaccine, value in vaccines.items():
                total_entries += 1
                
                # Skip if inventory value is NaN
                if math.isnan(value):
                    continue
                
                # Track zero vs non-zero values
                if value == 0:
                    zero_entries += 1
                else:
                    non_zero_entries += 1
                
                # Only proceed if we have valid price data
                if vaccine in avg_prices_per_period and year_int in avg_prices_per_period[vaccine]:
                    avg_price = avg_prices_per_period[vaccine][year_int]
                    h_value = h.get(vaccine, 0.01)  # Default to 0.01 if missing
                    
                    # Calculate the value
                    try:
                        calculated = (value * avg_price * h_value) / delta_value
                        
                        # Only keep valid results
                        if not math.isnan(calculated) and not math.isinf(calculated):
                            result[scenario][year][vaccine] = calculated
                            valid_entries += 1
                    except Exception as e:
                        print(f"Error calculating for {scenario}-{year}-{vaccine}: {e}")
        except Exception as e:
            print(f"Error processing year {year} in scenario {scenario}: {e}")

print(f"Total entries: {total_entries}")
print(f"Zero entries: {zero_entries}")
print(f"Non-zero entries: {non_zero_entries}")
print(f"Valid calculated entries: {valid_entries}")

# Properly sum scenario values with detailed tracking
scenario_sums_H = {}
for scenario, years in result.items():
    scenario_sum = 0
    year_count = 0
    non_zero_year_count = 0
    
    for year, vaccines in years.items():
        year_sum = 0
        vaccine_count = 0
        
        for vaccine, value in vaccines.items():
            year_sum += value
            vaccine_count += 1
        
        if year_sum > 0:
            non_zero_year_count += 1
            
        scenario_sum += year_sum
        year_count += 1
        
    scenario_sums_H[scenario] = scenario_sum
    if scenario_sum > 0:
        print(f"Scenario {scenario}: sum={scenario_sum:.2f}, from {year_count} years ({non_zero_year_count} with non-zero values)")

# Calculate weighted inventory cost
I_OBJ_Values = {}
total_weighted = 0
for k in probabilities:
    if k in scenario_sums_H:
        I_OBJ_Values[k] = probabilities[k] * scenario_sums_H[k]
        total_weighted += I_OBJ_Values[k]
    else:
        print(f"Warning: Key {k} not found in scenario_sums_H")
        I_OBJ_Values[k] = 0

I_OBJ_Value = sum(I_OBJ_Values.values())
print(f"Inventory Holding OBJ Value: {I_OBJ_Value}")

Total entries: 5950
Zero entries: 4142
Non-zero entries: 1808
Valid calculated entries: 5600
Scenario 5: sum=119522.07, from 10 years (9 with non-zero values)
Scenario 16: sum=119446.89, from 10 years (9 with non-zero values)
Scenario 20: sum=133326.06, from 10 years (9 with non-zero values)
Scenario 35: sum=114668.33, from 10 years (9 with non-zero values)
Scenario 12: sum=113625.02, from 10 years (8 with non-zero values)
Scenario 24: sum=133300.50, from 10 years (9 with non-zero values)
Scenario 28: sum=115620.47, from 10 years (8 with non-zero values)
Scenario 8: sum=115688.43, from 10 years (8 with non-zero values)
Scenario 17: sum=114809.84, from 10 years (9 with non-zero values)
Scenario 30: sum=119305.11, from 10 years (9 with non-zero values)
Scenario 23: sum=119410.30, from 10 years (9 with non-zero values)
Scenario 19: sum=115687.23, from 10 years (8 with non-zero values)
Scenario 22: sum=115639.39, from 10 years (8 with non-zero values)
Scenario 1: sum=134044.22, from 10 yea